In [5]:
from playwright.async_api import async_playwright, Error, TimeoutError
import pandas as pd
import time
import os


In [2]:
#Variables
host = "https://open.spotify.com/"
song_caracters = []
song_albunes = []
time_await_defaut = 1000
artist = 'Tyga'

In [158]:
async with async_playwright() as p:

    try:
        print('Configuramos el navegador.')
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        print(f'Redireccionamos al {host}')
        await page.goto(host, timeout=10000)
        await page.wait_for_timeout(time_await_defaut * 10)

    except TimeoutError:
        print(f'Timeout: El portal no cargo la url {host}')

    except Error as e:
        print('Error Playwright:', e)

    except Exception as e:
        print('Error inesperado:', e)

    print(f'Buscando a: {artist}')
    try:
        print(f'Ingresamos en el buscador: {artist}.')
        await page.locator('[role="combobox"]').click()
        await page.wait_for_timeout(time_await_defaut * 3)
        await page.locator('[role="combobox"]').fill(f'{artist}')
        await page.wait_for_timeout(time_await_defaut * 10)

        #obtenemos la verificacion del artista a buscar de una manera mas desglozada
        print('Valdiando que exista el artista a buscar.')
        card_results = page.locator('[data-testid="top-result-card"]')
        a_tag = card_results.locator('a').first
        artist_tag = a_tag.locator('div')
        name_artist = await artist_tag.first.inner_text()
        await artist_tag.click()

    except TimeoutError:
        print('Timeout durante la buscar del artista.')

    except Error as e:
        print('Error Playwright durante la busqueda del artista:', e)

    except Exception as e:
        print('Error inesperado durante la bsuqueda del artista:', e)


    #Obtenemos la lista de las canciones mas reproducidas del artista y sus atributos.
    print('Obtenemos las lista de canciones mas escuchadas.')
    try:
        await page.get_by_text('Ver más').click()
        await page.wait_for_timeout(time_await_defaut * 3)
        card_list_songs = page.locator('[role="presentation"] div[data-testid="tracklist-row"]')
        print(await card_list_songs.count())

        for i in range(await card_list_songs.count()):

            atributos = card_list_songs.nth(i).locator('[role=gridcell] div[data-encore-id="text"]')
            
            if await atributos.count() == 3:
                #for atributo in range(await atributos.count()):
                song_name = await atributos.nth(0).inner_text()
                song_num_repoduccion = await atributos.nth(1).inner_text()
                song_time = await atributos.nth(2).inner_text()
            
                song_caracters.append({
                    'artist': artist,
                    'name song': song_name.replace(',','').strip(),
                    'num reproduction': song_num_repoduccion.replace(',','').strip(),
                    'time duration': song_time.replace(',','').strip()
                })

            else:
                raise('El numero de atributos cambio ya no es 3, revisar el portal')

    except TimeoutError:
        print('Timeout: no apareció el elemento')

    except Error as e:
        print('Error Playwright:', e)

    except Exception as e:
        print('Error inesperado:', e)

    #Obtenemos la discografia del artista y sus atributos.
    print('Obtenemos toda la discografia.')
    try:

        page_discografia = page.locator('[aria-label="Discografía"] span[data-encore-id="text"]')
        await page_discografia.click()
        await page.wait_for_timeout(time_await_defaut * 3)
        await page.get_by_text("Fecha de lanzamiento").click()
        await page.wait_for_timeout(time_await_defaut * 3)
        await page.get_by_text("Cuadrícula").click()
        await page.wait_for_timeout(time_await_defaut * 5)

        all_discografia = page.locator('[role="presentation"] div[data-encore-id="card"]')
        for j in range(await all_discografia.count()):
            disc_name = await all_discografia.nth(j).locator('[data-encore-id="cardTitle"]').get_attribute('title')
            disc_type = await all_discografia.nth(j).locator('[data-encore-id="cardSubtitle"]').text_content()

            song_albunes.append({
                    'artist': artist,
                    'name album': disc_name.replace(',','').strip(),
                    'year': disc_type.split(' ')[0].replace(',','').strip(),
                    'type_disc': disc_type.split(' ')[-1].replace(',','').strip()
                })


    except TimeoutError:
        print('Timeout: no apareció el elemento')

    except Error as e:
        print('Error Playwright:', e)

    except Exception as e:
        print('Error inesperado:', e)

Configuramos el navegador.
Redireccionamos al https://open.spotify.com/
Buscando a: Tyga
Ingresamos en el buscador: Tyga.
Valdiando que exista el artista a buscar.
Obtenemos las lista de canciones mas escuchadas.
10
Obtenemos toda la discografia.


In [ ]:
df = pd.DataFrame(song_albunes)
df.to_csv(f'{artist}_albunes',encoding='utf8', index=True)


df = pd.DataFrame(song_caracters)
df.to_csv(f'{artist}_top_10',encoding='utf8', index=True)



In [ ]:
print('Generando archivo principal de artistas.')
path_fuente='/Users/axel/Documents/Portafolio'
archivos = [name for name in os.listdir(path_fuente) if name.endswith("top 10 of artist.csv")]
print(archivos)
for archivo in archivos:
    df_artist = pd.read_csv(f'{path_fuente}/{archivo}', encoding='utf-8', dtype=str)
    df_artist = df_artist[df_artist["artist"] != "artist"]
    df_artist['num reproduction'] = df_artist['num reproduction'].astype(int)

df_grouped = df_artist.groupby("artist").agg({
    "num reproduction": "sum"
}).reset_index()

columas_new_add = [ 
            'real_name' ,
            'musical_genre',
            'artist_type' ,
            'start_carrer' ,
            'end_carrer' ,
            'number_albums' ,
            'followers' ,
            'monthly_listeners']

df_grouped = df_grouped.assign(**{col: '' for col in columas_new_add})
df_grouped.to_csv('list_artist.csv', encoding='utf-8', index=False, header=True)



Generando archivo principal de artistas.
['top 10 of artist.csv']


,artist,num reproduction,real_name,musical_genre,artist_type,start_carrer,end_carrer,number_albums,followers,monthly_listeners
0,Billie Eilish,21760539808,,,,,,,,
1,Buscabulla,691873087,,,,,,,,
2,Hapax,2474339,,,,,,,,
3,Hocico,14699212,,,,,,,,
4,Kanye West,10889977591,,,,,,,,
5,Kendrick Lamar,17412690295,,,,,,,,
6,Kings of Leon,4773364600,,,,,,,,
7,Linea Aspera,22273475,,,,,,,,
8,Metallica,9059010240,,,,,,,,
9,Michael Jackson,9770564678,,,,,,,,


In [55]:
path_fuente='/Users/axel/Documents/Portafolio'
csv_artist = 'list_artist.csv'
csv_top_songs = 'top 10 of artist.csv'
csv_albunes = 'Albunes of artist.csv'


print(f'Leyendo archivo {csv_artist} para extraer index:')
archivos = [name for name in os.listdir(path_fuente) if name.endswith(csv_artist)]
for archivo in archivos:
    df_artist = pd.read_csv(f'{path_fuente}/{archivo}', encoding='utf-8', dtype=str)
    df_artist = df_artist.where(pd.notnull(df_artist), None)

    artist_name = list(df_artist['artist'])
    artist_map = {artist: i for i, artist in enumerate(artist_name)}

print(f'Leyendo archivo {csv_top_songs} para informacion y añadir index:')
archivos = [name for name in os.listdir(path_fuente) if name.endswith(csv_top_songs)]
for archivo in archivos:
    df_top = pd.read_csv(f'{path_fuente}/{archivo}', encoding='utf-8', dtype=str)
    df_top = df_top.where(pd.notnull(df_top), None)
    
    df_top['id artist'] = df_top['artist'].map(artist_map)

data = list(df_top.itertuples(index=False, name=None))

data

Leyendo archivo list_artist.csv para extraer index:
Leyendo archivo top 10 of artist.csv para informacion y añadir index:


[('Linea Aspera', 'Synapse', '7464611', '4:16', 7),
 ('Linea Aspera', 'Attica', '2054684', '3:38', 7),
 ('Linea Aspera', 'Eviction', '3204270', '4:03', 7),
 ('Linea Aspera', 'Malarone', '3003152', '4:11', 7),
 ('Linea Aspera', 'Fer-De-Lance', '1743070', '3:52', 7),
 ('Linea Aspera', 'Reunion', '1161786', '3:45', 7),
 ('Linea Aspera', 'Hinterland', '1138669', '3:48', 7),
 ('Linea Aspera', 'Lamanai', '1214756', '3:58', 7),
 ('Linea Aspera', 'Solar Flare', '810671', '5:45', 7),
 ('Linea Aspera', 'Redshift', '477806', '4:11', 7),
 ('Hocico', 'Sex Sick', '1943145', '5:08', 3),
 ('Hocico', 'Fed Up', '3188313', '4:41', 3),
 ('Hocico',
  'Und die Engel singen - Alas Caidas Remix by Hocico',
  '644764',
  '4:17',
  3),
 ('Hocico', 'I Want to Go to Hell', '2137045', '4:38', 3),
 ('Hocico', 'Hey Tú!', '237172', '4:12', 3),
 ('Hocico', 'Dark Paradigm', '59130', '6:02', 3),
 ('Hocico', 'Dead Trust', '1190953', '5:38', 3),
 ('Hocico', 'Ecos', '2409983', '5:57', 3),
 ('Hocico', 'Obscured', '1513595',